# Core claims and argument maps v04: resolve partial repeats

v03 restored scenarios and complements but the maps still repeated ideas that the strict "same idea" test (pass 1) does not merge, e.g. "The Fed faces a simple choice" next to "The Fed's choice is between a hike and risking a large bond spike". v04 adds a **second resolution pass** over the pass-1 canonical atoms (Opus, once, cached in `relations2.json`) with two further relations:

- **Subsumed**: atom X adds nothing beyond atom Y (a vaguer, partial or framing version). X is folded into Y.
- **Facets**: atoms that split one idea by subject, cause or conjunct (e.g. "Warsh / others at the Fed / the Trump admin understand bond risk"; three catalysts of one pricing shift). They become one statement with a composite wording, unless the author argues them separately.

Distinct numbers, horizons, scopes, subjects argued separately, and atoms in different scenarios are never folded. The mapping prompt and everything downstream (scenarios, complements, themed diagrams) are unchanged from v03 except that the statements are now the final ideas.

The comparison counts how many atoms in the v03 maps belong to the same final idea (repeats) against v04. Outputs: `outputs/argument_core_test_v04/` (`relations.json` is the pass-1 file copied from v02). Delete a file or folder to re-run it.

In [ ]:
import json
import re
import sys
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))

from incolor.argumentation.backends.claude_agent import ClaudeAgentBackend

OUT = ROOT / "outputs" / "argument_core_test_v04"
OUT.mkdir(parents=True, exist_ok=True)
PREV = ROOT / "outputs" / "argument_core_test_v03"
MODELS = ["haiku", "sonnet", "opus"]
CONDITIONS = ["nocore", "core"]
ROLES = ["premises", "conditions", "qualifiers", "concessions", "risks", "implications", "alternatives"]

DOC = re.sub(r"\A---.*?---\s*", "", (ROOT / "samples/BofA_Sept_FOMC_-_hike_or_bond_spike.txt").read_text(encoding="utf-8"), flags=re.S)
ATOMISE = ROOT / "outputs/atomise_v01/opus"
props = {
    m.group(1): m.group(2)
    for ln in (ATOMISE / "input.txt").read_text(encoding="utf-8").splitlines()
    if (m := re.match(r"\[(P\d+)\]\s+(.*)", ln))
}
ALL_ATOMS = {a["id"]: a for a in (json.loads(ln) for ln in (ATOMISE / "atoms.jsonl").read_text(encoding="utf-8").splitlines() if ln.strip())}
for a in ALL_ATOMS.values():
    a["pid"] = a["id"].split("-")[0]

# Core propositions: union of the Haiku and Opus votes
CORE_PROPS = sorted(
    {
        row["id"]
        for m in ("haiku", "opus")
        for row in json.loads((ROOT / f"outputs/claim_core_test_v02/{m}.json").read_text(encoding="utf-8"))["rows"]
        if row.get("core")
    }
)
print(len(props), "propositions,", len(ALL_ATOMS), "atoms; core propositions:", CORE_PROPS)

120 propositions, 269 atoms; core propositions: ['P005', 'P007', 'P009', 'P010']


In [2]:
RESOLVE_SYSTEM = """You resolve repeated ideas in a sell-side FX/rates research note. You get the full note, then its claims pre-extracted as atoms (IDs like P009-2, each with an extractor modality tag), grouped under the sentence they came from. The note often states the same idea more than once (key takeaways, body, summary).

Task: cluster atoms that express THE SAME IDEA, so each idea is represented once.

Two atoms are the same idea only if, in the context of this note, neither could be true while the other is false: same subject, same predicate, same polarity, same time and scope. Differences in wording, pronouns, tense, or trivial detail do not matter.

Do NOT merge:
- atoms that are merely related: cause and effect, premise and conclusion, part and whole, general and specific, or the same topic with different content (e.g. "hikes 25bp" vs "hikes 75bp in total");
- atoms of different modality class: an author's expectation, a plain assertion, a supposed condition (SUPPOSED) and a claim attributed to others are different even when the wording matches (e.g. EXPECTED "The Fed hikes 25bp" is not the same idea as SUPPOSED "The Fed hikes");
- atoms in different scenarios whose conditions or consequences differ.

For each cluster choose the canonical atom: the clearest, most complete wording, preferring the headline statement (key takeaways) when it is equally clear. List every member, including the canonical atom. Report only clusters with 2 or more members; atoms not listed stay on their own. Each atom belongs to at most one cluster.

Return ONE JSON object and nothing else (no code fences):
{"clusters": [{"canonical": "P005-1", "members": ["P005-1", "P135-1"], "why": "<=10 words"}]}"""


def atom_blocks() -> str:
    blocks = []
    for pid, text in props.items():
        group = [a for a in ALL_ATOMS.values() if a["pid"] == pid]
        if group:
            blocks.append(f"[{pid}] {text}\n" + "\n".join(f"  {a['id']} [{a['modality']}] {a['text']}" for a in group))
    return "\n".join(blocks)


def validate_clusters(data: dict) -> list[str]:
    errors, seen = [], Counter()
    for c in data.get("clusters", []):
        members = c.get("members", [])
        if len(members) < 2:
            errors.append(f"cluster {c.get('canonical')!r} has fewer than 2 members")
        if c.get("canonical") not in members:
            errors.append(f"canonical {c.get('canonical')!r} is not among its members")
        for x in members:
            if x not in ALL_ATOMS:
                errors.append(f"unknown atom {x!r}")
            seen[x] += 1
    return errors + [f"atom {x} is in {n} clusters" for x, n in seen.items() if n > 1]


def resolve() -> dict:
    path = OUT / "relations.json"
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))
    user = f"<document>\n{DOC}\n</document>\n\n<atoms>\n{atom_blocks()}\n</atoms>"
    backend = ClaudeAgentBackend(model="opus")
    raws, errors, data = [], [], {}
    for attempt in range(2):
        prompt = user if attempt == 0 else user + "\n\nYour previous answer had these errors; return the corrected full JSON object:\n- " + "\n- ".join(errors[:30]) + f"\n\nPrevious answer:\n{raws[-1]}"
        raws.append(backend.generate(RESOLVE_SYSTEM, prompt, {}))
        try:
            data = json.loads(raws[-1][raws[-1].index("{") : raws[-1].rindex("}") + 1])
            errors = validate_clusters(data)
        except ValueError as exc:
            data, errors = {}, [f"unparseable JSON: {exc}"]
        if not errors:
            break
    result = {"clusters": data.get("clusters", []), "errors": errors, "raw": raws, "usage": backend.usage}
    path.write_text(json.dumps(result, indent=1, ensure_ascii=False), encoding="utf-8")
    return result


# The Claude backend calls asyncio.run, so it must not run on the notebook's event-loop thread
with ThreadPoolExecutor(1) as pool:
    relations = pool.submit(resolve).result()
assert not relations["errors"], relations["errors"]

# cluster_of maps every atom to its canonical atom; atoms (canonical only) is what the map builders see
cluster_of = {i: i for i in ALL_ATOMS}
for c in relations["clusters"]:
    for x in c["members"]:
        cluster_of[x] = c["canonical"]
atoms = {}
for i, a in ALL_ATOMS.items():
    if cluster_of[i] == i:
        members = [x for x, k in cluster_of.items() if k == i]
        atoms[i] = {**a, "members": members, "pids": {ALL_ATOMS[x]["pid"] for x in members}}
CORE_ATOMS = {i for i, a in atoms.items() if a["pids"] & set(CORE_PROPS)}

merged = [c for c in relations["clusters"]]
print(f"{len(ALL_ATOMS)} atoms -> {len(atoms)} ideas ({len(merged)} merged clusters, {len(ALL_ATOMS) - len(atoms)} atoms folded away); {len(CORE_ATOMS)} core ideas")
cost = sum(u.get("cost_usd") or 0 for u in relations["usage"])
print(f"resolution cost ${cost:.3f}")
for c in merged:
    print(f"  {c['canonical']}: {ALL_ATOMS[c['canonical']]['text']}  <- {', '.join(x for x in c['members'] if x != c['canonical'])}")

269 atoms -> 247 ideas (13 merged clusters, 22 atoms folded away); 7 core ideas
resolution cost $0.232
  P005-1: The Fed hikes 25bp.  <- P135-1
  P022-1: The SEP shows 50bps of total hikes in '26.  <- P136-1
  P016-1: The Fed faces a simple choice at the Sept FOMC.  <- P007-1, P137-1, P098-1
  P016-2: The Fed's choice at the Sept FOMC is between a hike and risking a large bond spike.  <- P007-2, P137-2, P098-2
  P019-1: Bond price action, including post Friday's CPI, shows the bond likes orthodoxy.  <- P008-1
  P009-1: Warsh delivers a hawkish hike.  <- P138-1, P102-1, P085-1
  P009-2: Warsh's hawkish hike supports higher front end rates.  <- P138-2, P103-3
  P009-3: Warsh's hawkish hike supports twist flattening of the UST curve.  <- P138-3, P103-4, P081-1
  P010-1: The USD outcome is determined by the Fed's ability to "out-hawk" market expectations and other G10 central banks.  <- P139-1
  P013-1: Rates price the Sep hike with approximately 85% likelihood.  <- P079-1
  P096-12: Treas

In [3]:
REDUNDANCY_SYSTEM = """You remove redundancy between the claims of a sell-side FX/rates research note. You get the full note, then a list of claims (atoms, IDs like P016-2, each with an extractor modality tag, grouped under the sentence they came from). Strict duplicates are already merged, but some atoms still repeat what another atom says, wholly or in part. Find them.

Two relations:

1. SUBSUMED: atom X adds nothing that atom Y does not already say or entail in this note. X is a vaguer, partial, or framing version of Y. Test: a reader who has Y learns nothing more from X. Example: "The Fed faces a simple choice at the Sept FOMC" is subsumed by "The Fed's choice at the Sept FOMC is between a hike and risking a large bond spike". Y is the more specific, more informative atom. If X adds anything (another subject, cause, number, horizon, scope, or a distinct judgement), it is NOT subsumed.

2. FACETS: atoms that split ONE idea by subject, object, cause or conjunct: same predicate, same modality, same or adjacent sentence, differing only in one slot, and not argued separately by the author. Examples: "Warsh understands bond risk with a Sept hold" / "Others at the Fed understand ..." / "The Trump admin understands ..."; "The sharp shift in Fed pricing was catalyzed by the speech" / "... by stronger labor data" / "... by stronger inflation data"; "Warsh's remarks have raised the cost of disappointing expectations" / "Current market pricing has raised the same cost". Merge them into one group with a composite `text`: one sentence that states the whole idea using the atoms' own words and preserving every hedge word.

Never fold:
- atoms of different modality class (an expectation, a plain assertion, a supposed condition, a claim attributed to others);
- atoms related only as cause and effect, premise and conclusion, or data and the judgement it supports;
- atoms with different numbers, horizons or scopes (e.g. "2Y rises 5-10bps" and "30Y falls 5-10bps");
- facets the author treats separately (different roles, different supporting reasons);
- atoms in different scenarios whose conditions or consequences differ.

Be conservative: fold only when the redundancy is clear.

Return ONE JSON object and nothing else (no code fences):
{"subsumed": [{"atom": "P016-1", "by": "P016-2", "why": "<=10 words"}],
 "facets": [{"canonical": "P097-1", "members": ["P097-1", "P097-2", "P097-3"], "text": "...", "why": "<=10 words"}]}
Each atom may appear in at most one entry as the folded atom or facet member. Use empty lists if there is nothing to fold."""


def validate_redundancy(data: dict) -> list[str]:
    errors, seen = [], Counter()
    sub = data.get("subsumed", [])
    for s in sub:
        if s.get("atom") not in atoms or s.get("by") not in atoms or s.get("atom") == s.get("by"):
            errors.append(f"bad subsumed entry {s!r}")
        seen[s.get("atom")] += 1
    for f in data.get("facets", []):
        members = f.get("members", [])
        if len(members) < 2 or f.get("canonical") not in members or not f.get("text"):
            errors.append(f"bad facet group {f.get('canonical')!r}")
        for x in members:
            if x not in atoms:
                errors.append(f"unknown atom {x!r}")
            seen[x] += 1
    errors += [f"atom {x} is folded {n} times" for x, n in seen.items() if n > 1]
    # a chain of 'by' links must end at an unfolded atom
    by = {s["atom"]: s["by"] for s in sub if "atom" in s and "by" in s}
    for x in by:
        hops, y = 0, x
        while y in by and hops <= len(by):
            y, hops = by[y], hops + 1
        if y in by:
            errors.append(f"subsumed cycle through {x}")
    return errors


def pass2_blocks() -> str:
    blocks = []
    for pid, text in props.items():
        group = [a for a in atoms.values() if a["pid"] == pid]
        if group:
            blocks.append(f"[{pid}] {text}\n" + "\n".join(f"  {a['id']} [{a['modality']}] {a['text']}" for a in group))
    return "\n".join(blocks)


def redundancy() -> dict:
    path = OUT / "relations2.json"
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))
    user = f"<document>\n{DOC}\n</document>\n\n<atoms>\n{pass2_blocks()}\n</atoms>"
    backend = ClaudeAgentBackend(model="opus")
    raws, errors, data = [], [], {}
    for attempt in range(2):
        prompt = user if attempt == 0 else user + "\n\nYour previous answer had these errors; return the corrected full JSON object:\n- " + "\n- ".join(errors[:30]) + f"\n\nPrevious answer:\n{raws[-1]}"
        raws.append(backend.generate(REDUNDANCY_SYSTEM, prompt, {}))
        try:
            data = json.loads(raws[-1][raws[-1].index("{") : raws[-1].rindex("}") + 1])
            errors = validate_redundancy(data)
        except ValueError as exc:
            data, errors = {}, [f"unparseable JSON: {exc}"]
        if not errors:
            break
    result = {"subsumed": data.get("subsumed", []), "facets": data.get("facets", []), "errors": errors, "raw": raws, "usage": backend.usage}
    path.write_text(json.dumps(result, indent=1, ensure_ascii=False), encoding="utf-8")
    return result


# atoms here is still the pass-1 canonical set; the final set replaces it below
with ThreadPoolExecutor(1) as pool:
    relations2 = pool.submit(redundancy).result()
assert not relations2["errors"], relations2["errors"]

by = {s["atom"]: s["by"] for s in relations2["subsumed"]}
facet_of = {x: f["canonical"] for f in relations2["facets"] for x in f["members"]}


def final_of(x: str) -> str:
    while x in by or (x in facet_of and facet_of[x] != x):
        x = by[x] if x in by else facet_of[x]
    return x


pass1_of = dict(cluster_of)  # every original atom -> pass-1 canonical
cluster_of = {i: final_of(k) for i, k in pass1_of.items()}
facet_text = {f["canonical"]: f["text"] for f in relations2["facets"]}
atoms1, atoms = atoms, {}
for i, a in ALL_ATOMS.items():
    if cluster_of[i] == i:
        members = [x for x, k in cluster_of.items() if k == i]
        facets = [x for x in members if x != i and facet_of.get(pass1_of[x]) == i]
        atoms[i] = {**a, "text": facet_text.get(i, a["text"]), "members": members, "facets": facets, "pids": {ALL_ATOMS[x]["pid"] for x in members}}
CORE_ATOMS = {i for i, a in atoms.items() if a["pids"] & set(CORE_PROPS)}

print(f"{len(ALL_ATOMS)} atoms -> {len(atoms1)} after pass 1 -> {len(atoms)} final ideas; {len(CORE_ATOMS)} core ideas")
print(f"pass 2 cost ${sum(u.get('cost_usd') or 0 for u in relations2['usage']):.3f}")
print("\nSubsumed:")
for s in relations2["subsumed"]:
    print(f"  {s['atom']}: {ALL_ATOMS[s['atom']]['text']}\n    -> {s['by']}: {ALL_ATOMS[s['by']]['text']}")
print("\nFacets:")
for f in relations2["facets"]:
    print(f"  {', '.join(f['members'])}\n    => {f['text']}")

269 atoms -> 247 after pass 1 -> 214 final ideas; 5 core ideas
pass 2 cost $0.291

Subsumed:
  P016-1: The Fed faces a simple choice at the Sept FOMC.
    -> P016-2: The Fed's choice at the Sept FOMC is between a hike and risking a large bond spike.
  P041-2: Gov. Waller dissents.
    -> P022-2: Waller dissents dovishly.
  P073-2: Warsh made tweaks to Warsh's communication style at Jackson Hole.
    -> P073-1: The author is curious whether Warsh retains the tweaks Warsh made to Warsh's communication style at Jackson Hole.
  P115-2: Chair Warsh attempted a course correction at Jackson Hole.
    -> P115-3: Chair Warsh's attempted course correction at Jackson Hole was bullish.
  P115-4: The August labor report was strong.
    -> P115-5: The strong August labor report was bullish.
  P115-6: The Fed's Williams made dovish comments.
    -> P115-8: The dovish comments from the Fed's Williams and Waller were bearish.
  P115-7: The Fed's Waller made dovish comments.
    -> P115-8: The dovish co

In [4]:
SYSTEM = """You are a financial-research argument analyst. You get a sell-side research note in full, then its claims pre-extracted as atoms (IDs like P009-2, each with the extractor's modality tag), grouped under the sentence they came from. Atoms that restate, or are fully covered by, another atom have already been folded into it, and atoms that are facets of one idea (split by subject or cause) are merged into one atom with a composite wording: each idea appears once. An "also stated as" note shows where else the note repeats it, and a "merged facets" note lists the atoms that were merged into a composite.

Task: rebuild the CORE ARGUMENTS of the note as an argument map over the atoms, with SCENARIOS where the author reasons conditionally. Do not use every atom: use only the atoms that do argumentative work for the author's main conclusions, and leave out the rest (background, data not needed as support, side scenarios, detail).

# Arguments

An argument has ONE conclusion (an atom ID) and the members that bear on it, each an atom ID, a scenario ID (SC###) or the ID of another argument (A###, meaning that argument's conclusion is itself a member, supported by its own members). Roles:
- premises: reasons or evidence the author gives for the conclusion
- conditions: what the conclusion depends on (if/unless/assuming)
- qualifiers: scope or strength limits on the conclusion
- concessions: points the author grants that cut against the conclusion
- risks: what could make the conclusion fail or hurt
- implications: what follows from the conclusion
- alternatives: the author's stated alternative outcomes

# Scenarios and complements

Much of the note's reasoning is conditional: "if the Fed does not hike ... it risks a sharp UST move", "assuming a hike ...", a hawkish presser versus a dovish presser, a bullish versus a bearish USD case. Atoms tagged SUPPOSED are the supposed conditions (the if/assuming/unless part); the other atoms of the same sentence, and of the sentences that follow in the same passage, are what the author says follows from them.

- A SCENARIO is a supposed situation plus what the author says follows from it: `assumptions` (SUPPOSED atoms) and `implications` (the consequent atoms). The same supposition is often repeated in several sentences (e.g. "The Fed hikes" in P038, P041, P045, P058): cite ONE atom as the assumption and gather the consequents of all its repetitions under `implications`.
- Scenarios that answer the same question differently are COMPLEMENTS: the hike case and the hold case, a hawkish and a dovish presser, a bullish and a bearish USD case, Warsh as at Jackson Hole and as at the July FOMC. Give each scenario the IDs of its complements under `complements` (a scenario may have several, and the relation is mutual).
- Frame complements as one choice. Write an argument whose conclusion is the atom that poses the choice (e.g. "The Fed's choice at the Sept FOMC is between a hike and risking a large bond spike") and list the complementary scenarios under that argument's `alternatives`. Say which branch the author expects by making that branch's expected outcome the conclusion of its own argument, with the framing argument or the scenario as a premise.
- Other scenarios are members of the argument they bear on, in whichever role fits (usually conditions, implications or risks).
- Every scenario has at least one assumption and at least one implication, and is a member of exactly one argument. Its atoms appear nowhere else in the map: a consequent that follows from the supposition belongs in the scenario, not also as a premise elsewhere.
- Use a scenario only for a supposed situation with consequences. A bare condition on one conclusion with no further consequences is a `conditions` member of that argument.

# Rules

- Use the whole note to see how the author argues; link members only where the author states or clearly structures the link. Do not invent links.
- Use atom IDs exactly as given. Never rewrite atom text.
- Each atom appears at most once in the whole map (as one argument's conclusion, one member of one argument, or one scenario assumption or implication).
- Never restate an idea: a point that is already a conclusion, member or scenario atom must not reappear in another role, argument or scenario. Use the "also stated as" locations only to understand where the author argues the point.
- Each argument is a member of at most one other argument; no cycles. Top-level arguments are those nobody references.
- Every argument has at least one premise.
- Build hierarchy: if a premise has reasons of its own, make it the conclusion of its own argument and reference that argument's ID as the member.
- Summaries: one sentence, at most 30 words, using only claims of the member atoms.

Return ONE JSON object and nothing else (no code fences), keys in this order:
{"arguments": [{"id": "A001", "summary": "...", "conclusion": "P016-2", "premises": ["P019-1"], "conditions": [], "qualifiers": [], "concessions": [], "risks": [], "implications": [], "alternatives": ["SC001", "SC002"]}],
 "scenarios": [{"id": "SC001", "label": "<=8 words", "assumptions": ["P017-1"], "implications": ["P017-2"], "complements": ["SC002"]}]}
Number arguments A001, A002, ... and scenarios SC001, SC002, ... in the order you list them. Use "scenarios": [] if there are none."""

CORE_BLOCK = """
The author's headline (core) claims have been identified in a prior pass and are flagged CORE in the atom list. Build the map so that each top-level argument concludes one of the core claims, or the core claims are reached through the hierarchy. Include the scenarios that frame or bear on a core claim (for example the hike and no-hike cases around a core hike call). Reasoning that does not bear on a core claim need not appear."""


def build_user(condition: str) -> str:
    blocks = []
    for pid, text in props.items():
        group = [a for a in atoms.values() if a["pid"] == pid]
        if not group:
            continue
        lines = []
        for a in group:
            core = " [CORE]" if condition == "core" and a["id"] in CORE_ATOMS else ""
            also = [f"{x} ({ALL_ATOMS[x]['pid']})" for x in a["members"] if x != a["id"] and x not in a["facets"]]
            note = f"  (also stated as {', '.join(also)})" if also else ""
            if a["facets"]:
                note += f"  (merged facets: {', '.join(a['facets'])})"
            lines.append(f"  {a['id']} [{a['modality']}]{core} {a['text']}{note}")
        blocks.append(f"[{pid}] {text}\n" + "\n".join(lines))
    extra = CORE_BLOCK if condition == "core" else ""
    return f"<document>\n{DOC}\n</document>\n\n<atoms>\n" + "\n".join(blocks) + f"\n</atoms>\n{extra}"


def parse(raw: str) -> dict:
    m = json.loads(raw[raw.index("{") : raw.rindex("}") + 1])
    # Complements are mutual: add the reverse link where the model gave only one direction
    scen = {s.get("id"): s for s in m.get("scenarios", [])}
    for s in scen.values():
        for c in s.get("complements", []):
            if c in scen and c != s["id"] and s["id"] not in scen[c].setdefault("complements", []):
                scen[c]["complements"].append(s["id"])
    return m


def validate(m: dict) -> list[str]:
    """Structural checks on one argument map; returns error strings."""
    errors: list[str] = []
    args = {a.get("id"): a for a in m.get("arguments", [])}
    scen = {s.get("id"): s for s in m.get("scenarios", [])}
    used: Counter = Counter()
    refs: Counter = Counter()
    srefs: Counter = Counter()
    for a in args.values():
        concl = a.get("conclusion")
        if concl not in atoms:
            errors.append(f"{a.get('id')}: conclusion {concl!r} is not an atom ID")
        else:
            used[concl] += 1
        members = [x for r in ROLES for x in a.get(r, [])]
        if not a.get("premises"):
            errors.append(f"{a.get('id')}: no premises")
        for x in members:
            if x in atoms:
                used[x] += 1
            elif x in args:
                refs[x] += 1
                if x == a.get("id"):
                    errors.append(f"{x}: references itself")
            elif x in scen:
                srefs[x] += 1
            else:
                errors.append(f"{a.get('id')}: unknown member {x!r}")
    for s in scen.values():
        sid = s.get("id")
        if not s.get("assumptions"):
            errors.append(f"{sid}: no assumptions")
        if not s.get("implications"):
            errors.append(f"{sid}: no implications")
        for x in [*s.get("assumptions", []), *s.get("implications", [])]:
            if x in atoms:
                used[x] += 1
            else:
                errors.append(f"{sid}: unknown atom {x!r}")
        for c in s.get("complements", []):
            if c not in scen or c == sid:
                errors.append(f"{sid}: bad complement {c!r}")
        if srefs[sid] != 1:
            errors.append(f"scenario {sid} is a member of {srefs[sid]} arguments (must be exactly 1)")
    errors += [f"atom {i} used {n} times" for i, n in used.items() if n > 1]
    errors += [f"argument {i} referenced {n} times" for i, n in refs.items() if n > 1]
    # cycle check on argument references
    state: dict[str, int] = {}

    def visit(i: str) -> None:
        if state.get(i) == 2 or i not in args:
            return
        if state.get(i) == 1:
            errors.append(f"cycle through {i}")
            return
        state[i] = 1
        for r in ROLES:
            for x in args[i].get(r, []):
                if x in args:
                    visit(x)
        state[i] = 2

    for i in args:
        visit(i)
    return errors


def run(job: tuple[str, str]) -> dict:
    condition, model = job
    folder = OUT / condition / model
    path = folder / "argument_map.json"
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))
    folder.mkdir(parents=True, exist_ok=True)
    user = build_user(condition)
    backend = ClaudeAgentBackend(model=model)
    raws, errors, m = [], [], {}
    for attempt in range(2):
        prompt = user if attempt == 0 else user + f"\n\nYour previous answer had these errors; return a corrected full JSON object:\n- " + "\n- ".join(errors[:30]) + f"\n\nPrevious answer:\n{raws[-1]}"
        raws.append(backend.generate(SYSTEM, prompt, {}))
        (folder / f"raw{attempt + 1}.txt").write_text(raws[-1], encoding="utf-8")
        try:
            m = parse(raws[-1])
            errors = validate(m)
        except ValueError as exc:  # json.JSONDecodeError is a ValueError
            m, errors = {}, [f"unparseable JSON: {exc}"]
        if not errors:
            break
    result = {"map": m, "errors": errors, "attempts": len(raws), "usage": backend.usage}
    path.write_text(json.dumps(result, indent=1, ensure_ascii=False), encoding="utf-8")
    return result


jobs = [(c, m) for c in CONDITIONS for m in MODELS]
with ThreadPoolExecutor(len(jobs)) as pool:
    runs = dict(zip(jobs, pool.map(run, jobs)))

for (c, m), r in runs.items():
    cost = sum(u.get("cost_usd") or 0 for u in r["usage"])
    mp = r["map"]
    print(f"{c:7} {m:7} args={len(mp.get('arguments', []))} scenarios={len(mp.get('scenarios', []))} attempts={r['attempts']} errors={len(r['errors'])} ${cost:.3f}")

    for e in r["errors"][:5]:        print("   -", e)

nocore  haiku   args=14 scenarios=6 attempts=1 errors=0 $0.026
nocore  sonnet  args=10 scenarios=9 attempts=1 errors=0 $0.138
nocore  opus    args=11 scenarios=6 attempts=1 errors=0 $0.375
core    haiku   args=7 scenarios=6 attempts=1 errors=0 $0.020
core    sonnet  args=6 scenarios=6 attempts=1 errors=0 $0.116
core    opus    args=10 scenarios=8 attempts=1 errors=0 $0.326


In [5]:
from itertools import combinations


def scenario_atoms(m: dict) -> set[str]:
    return {x for s in m.get("scenarios", []) for r in ("assumptions", "implications") for x in s.get(r, [])}


def analyse(m: dict) -> dict:
    args = {a["id"]: a for a in m.get("arguments", []) if isinstance(a, dict) and "id" in a}
    scen = {s["id"]: s for s in m.get("scenarios", []) if isinstance(s, dict) and "id" in s}
    referenced = {x for a in args.values() for r in ROLES for x in a.get(r, []) if x in args}
    roots = [i for i in args if i not in referenced]
    used = {a.get("conclusion") for a in args.values()} | {x for a in args.values() for r in ROLES for x in a.get(r, [])}
    used = {x for x in used | scenario_atoms(m) if x in atoms}
    memo: dict[str, int] = {}

    def depth(i: str, seen=()) -> int:
        if i in seen:
            return 0
        if i not in memo:
            kids = [x for r in ROLES for x in args[i].get(r, []) if x in args]
            memo[i] = 1 + max((depth(k, (*seen, i)) for k in kids), default=0)
        return memo[i]

    root_ideas = {args[i].get("conclusion") for i in roots} & set(atoms)
    concl_ideas = {a.get("conclusion") for a in args.values()} & set(atoms)
    # Proposition ids include those of every restatement folded into the idea
    pid = lambda s: {p for x in s if x in atoms for p in atoms[x]["pids"]}
    pairs = {tuple(sorted((s["id"], c))) for s in scen.values() for c in s.get("complements", []) if c in scen}
    # A pair is framed when one argument holds both scenarios as members
    framed = {p for p in pairs for a in args.values() if all(x in [y for r in ROLES for y in a.get(r, [])] for x in p)}
    return {
        "args": len(args),
        "roots": len(roots),
        "depth": max((depth(i) for i in args), default=0),
        "scenarios": len(scen),
        "pairs": len(pairs),
        "framed": len(framed),
        "ideas_used": used,
        "props_used": pid(used),
        "root_ideas": root_ideas,
        "root_props": pid(root_ideas),
        "concl_props": pid(concl_ideas),
    }


rows = []
info = {}
for (c, m), r in runs.items():
    x = analyse(r["map"])
    info[(c, m)] = x
    core_used = set(CORE_PROPS) & x["props_used"]
    rows.append(
        {
            "cond": c,
            "model": m,
            "errors": len(r["errors"]),
            "args": x["args"],
            "roots": x["roots"],
            "depth": x["depth"],
            "scenarios": x["scenarios"],
            "compl_pairs": x["pairs"],
            "pairs_framed": x["framed"],
            "ideas_used": len(x["ideas_used"]),
            "%ideas": round(100 * len(x["ideas_used"]) / len(atoms)),
            "core_props_in_map": f"{len(core_used)}/{len(CORE_PROPS)}",
            "core_props_as_root": f"{len(set(CORE_PROPS) & x['root_props'])}/{len(CORE_PROPS)}",
            "root_ideas_that_are_core": f"{len(x['root_ideas'] & CORE_ATOMS)}/{len(x['root_ideas'])}",
            "cost": round(sum(u.get('cost_usd') or 0 for u in r["usage"]), 3),
        }
    )
metrics = pd.DataFrame(rows)
print(metrics.to_string(index=False))


def jac(a: set, b: set) -> float:
    return len(a & b) / len(a | b) if a | b else 1.0


print("\nCross-model agreement (mean pairwise Jaccard)")
for c in CONDITIONS:
    pairs = list(combinations(MODELS, 2))
    for key in ("props_used", "root_ideas"):
        v = sum(jac(info[(c, a)][key], info[(c, b)][key]) for a, b in pairs) / len(pairs)
        print(f"  {c:7} {key:11}: {v:.2f}")

  cond  model  errors  args  roots  depth  scenarios  compl_pairs  pairs_framed  ideas_used  %ideas core_props_in_map core_props_as_root root_ideas_that_are_core  cost
nocore  haiku       0    14      3      4          6            3             3          83      39               4/4                2/4                      2/3 0.026
nocore sonnet       0    10      5      2          9            6             6          82      38               4/4                3/4                      3/5 0.138
nocore   opus       0    11      3      3          6            3             3          93      43               4/4                3/4                      3/3 0.375
  core  haiku       0     7      3      3          6            3             3          75      35               4/4                3/4                      3/3 0.020
  core sonnet       0     6      2      4          6            3             3          41      19               4/4                2/4                      2/

In [6]:
# v03 (strict duplicates only) vs v04 (partial repeats resolved)
def stats(m: dict) -> dict:
    args = [a for a in m.get("arguments", []) if isinstance(a, dict) and "id" in a]
    ids = {a["id"] for a in args}
    referenced = {x for a in args for r in ROLES for x in a.get(r, []) if x in ids}
    used = [a["conclusion"] for a in args] + [x for a in args for r in ROLES for x in a.get(r, [])] + list(scenario_atoms(m))
    used = [x for x in used if x in ALL_ATOMS]
    ideas = {cluster_of[x] for x in used}
    roots = {cluster_of[a["conclusion"]] for a in args if a["id"] not in referenced and a["conclusion"] in ALL_ATOMS}
    return {"args": len(args), "scen": len(m.get("scenarios", [])), "atoms": len(set(used)), "ideas": ideas, "roots": roots}


cmp_rows, per_cell = [], {}
for version, base in (("v03", PREV), ("v04", OUT)):
    for c in CONDITIONS:
        for m in MODELS:
            s = stats(json.loads((base / c / m / "argument_map.json").read_text(encoding="utf-8"))["map"])
            per_cell[(version, c, m)] = s
            cmp_rows.append(
                {
                    "version": version,
                    "cond": c,
                    "model": m,
                    "args": s["args"],
                    "scenarios": s["scen"],
                    "atoms_used": s["atoms"],
                    "distinct_ideas": len(s["ideas"]),
                    "repeated_atoms": s["atoms"] - len(s["ideas"]),
                }
            )
cmp = pd.DataFrame(cmp_rows)
print(cmp.to_string(index=False))
print("\nTotals per version (repeated_atoms = atoms in a map that belong to an idea already in that map):")
print(cmp.groupby("version")[["args", "scenarios", "atoms_used", "distinct_ideas", "repeated_atoms"]].sum().to_string())

print("\nIdea-level agreement across models (mean pairwise Jaccard)")
for version in ("v03", "v04"):
    for c in CONDITIONS:
        pairs = list(combinations(MODELS, 2))
        for key in ("ideas", "roots"):
            v = sum(jac(per_cell[(version, c, a)][key], per_cell[(version, c, b)][key]) for a, b in pairs) / len(pairs)
            print(f"  {version} {c:7} {key:5}: {v:.2f}")

version   cond  model  args  scenarios  atoms_used  distinct_ideas  repeated_atoms
    v03 nocore  haiku     8          7          74              68               6
    v03 nocore sonnet     8          8          55              54               1
    v03 nocore   opus    18         10         129             116              13
    v03   core  haiku    10          7          77              71               6
    v03   core sonnet     8          8          63              60               3
    v03   core   opus     8          7          91              82               9
    v04 nocore  haiku    14          6          83              83               0
    v04 nocore sonnet    10          9          82              82               0
    v04 nocore   opus    11          6          93              93               0
    v04   core  haiku     7          6          75              75               0
    v04   core sonnet     6          6          41              41               0
    

In [ ]:
import yaml

from incolor.argumentation.diagram import ArgumentMapDiagram

# Atom-level types from claim_core_test (opus), mapped onto the standard theme's claim types
THEME_TYPE = {"FACT": "OBSERVATION", "VIEW": "VIEW", "FORECAST": "FORECAST", "PREDICTION": "PREDICTION", "ASSUMPTION": "SUPPOSITION"}
# Atomise v01 modalities mapped onto the theme's four
THEME_MODALITY = {
    "ASSERTED": "ASSERTED",
    "ATTRIBUTED": "ASSERTED",
    "EXPECTED": "HEDGED",
    "HEDGED": "HEDGED",
    "SPECULATIVE": "SPECULATIVE",
    "SUPPOSED": "HYPOTHETICAL",
}
atom_type = {
    r["id"]: THEME_TYPE.get(r["claim_type"].upper(), "VIEW")
    for r in json.loads((ROOT / "outputs/claim_core_test/opus.json").read_text(encoding="utf-8"))["rows"]
}


def to_theme_map(m: dict) -> dict:
    """Convert a run's map to the samples argument-map schema (atoms become statements)."""
    args = [a for a in m.get("arguments", []) if isinstance(a, dict) and "id" in a]
    scen = [s for s in m.get("scenarios", []) if isinstance(s, dict) and "id" in s]
    used = {a["conclusion"] for a in args} | {x for a in args for r in ROLES for x in a.get(r, []) if x in atoms} | scenario_atoms(m)
    statements = [
        {
            "id": i,
            "brief": a["text"],
            "statement": a["text"],
            "claim_type": atom_type.get(i, "VIEW"),
            "modality": THEME_MODALITY[a["modality"]],
            "evidence": [i],
        }
        for i, a in atoms.items()
        if i in used
    ]
    arguments = [
        {"id": a["id"], "summary": a.get("summary", ""), "conclusion": a["conclusion"], **{r: a.get(r, []) for r in ROLES}}
        for a in args
    ]
    scenarios = [
        {"id": s["id"], "label": s.get("label", s["id"]), "assumptions": s.get("assumptions", []), "implications": s.get("implications", []), "complements": s.get("complements", [])}
        for s in scen
    ]
    return {"statements": statements, "arguments": arguments, "scenarios": scenarios}


def with_complements(mermaid: str, scenarios: list[dict]) -> str:
    """Join complementary scenarios with a dotted edge; the theme has no scenario-to-scenario edge."""
    pairs = sorted({tuple(sorted((s["id"], c))) for s in scenarios for c in s.get("complements", [])})
    edges = [f"    {a} <-. Complements .-> {b}" for a, b in pairs]
    lines = mermaid.split("\n")
    at = next((n for n, ln in enumerate(lines) if ln.startswith("    classDef")), len(lines))
    return "\n".join([*lines[:at], *edges, "", *lines[at:]]) if edges else mermaid


for (c, mdl), r in runs.items():
    themed = to_theme_map(r["map"])
    folder = OUT / c / mdl
    (folder / "argument_map.yaml").write_text(yaml.safe_dump(themed, sort_keys=False, allow_unicode=True), encoding="utf-8")
    source = with_complements(ArgumentMapDiagram(themed).to_mermaid(), themed["scenarios"])
    (folder / "diagram.mmd").write_text(source, encoding="utf-8")
print("wrote", len(runs), "themed diagram.mmd + argument_map.yaml files under", OUT)


def outline(c: str, mdl: str) -> str:
    m = runs[(c, mdl)]["map"]
    args = {a["id"]: a for a in m.get("arguments", []) if isinstance(a, dict) and "id" in a}
    scen = {s["id"]: s for s in m.get("scenarios", []) if isinstance(s, dict) and "id" in s}
    referenced = {x for a in args.values() for r in ROLES for x in a.get(r, []) if x in args}
    out: list[str] = []
    text = lambda x: atoms.get(x, {}).get("text", "?")
    tag = lambda x: "*" if x in CORE_ATOMS else " "

    def show(i: str, level: int, role: str = "") -> None:
        a = args[i]
        c_id = a["conclusion"]
        out.append(f"{'    ' * level}{tag(c_id)}{role + ': ' if role else ''}{c_id} {text(c_id)}")
        for r in ROLES:
            for x in a.get(r, []):
                pad = "    " * (level + 1)
                if x in args:
                    show(x, level + 1, r)
                elif x in scen:
                    s = scen[x]
                    comp = f" (complements {', '.join(s.get('complements', []))})" if s.get("complements") else ""
                    out.append(f"{pad} {r}: [{x}] SCENARIO {s.get('label', '')}{comp}")
                    out.extend(f"{pad}      IF   {tag(y)}{y} {text(y)}" for y in s.get("assumptions", []))
                    out.extend(f"{pad}      THEN {tag(y)}{y} {text(y)}" for y in s.get("implications", []))
                else:
                    out.append(f"{pad}{tag(x)}{r}: {x} {text(x)}")

    for i in args:
        if i not in referenced:
            show(i, 0)
            out.append("")
    return "\n".join(out)

wrote 6 themed diagram.mmd + argument_map.yaml files under c:\Users\angel.adeoye\dev\financial_research\outputs\argument_core_test_v04


In [8]:
# * = atom of a core proposition
VIEW = [("nocore", "opus"), ("core", "opus")]
for c, mdl in VIEW:
    print(f"===== {c} / {mdl} =====")
    print(outline(c, mdl))

===== nocore / opus =====
*P005-1 The Fed hikes 25bp.
    *premises: P016-2 The Fed's choice at the Sept FOMC is between a hike and risking a large bond spike.
         premises: P019-1 Bond price action, including post Friday's CPI, shows the bond likes orthodoxy.
         premises: P020-1 Orthodoxy is to use rate hikes to slow inflation and quell long end rate spike.
         premises: P097-1 Warsh, others at the Fed, and the Trump admin understand bond risk with a Sept hold.
             premises: P099-1 The Trump admin's choice is between tolerating modestly higher front end rates and materially higher long end rates.
             premises: P100-1 Bessent and Warsh effectively explain to Trump the tradeoff between tolerating modestly higher front end rates and materially higher long end rates.
             premises: P101-2 Trump understands the risks of bond un-anchoring and its market implications.
             concessions: P101-1 Trump does not like the choice set between tolerat